## Imports

In [2]:
import os
from pathlib import Path
import datetime

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl 
import numpy as np
from sklearn.linear_model import ElasticNet, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor

import kaggle_evaluation.default_inference_server

## Project Directory Structure

In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hull-tactical-market-prediction/train.csv
/kaggle/input/hull-tactical-market-prediction/test.csv
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_inference_server.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py
/kaggl

## Configurations

In [4]:
# ============ PATHS ============
DATA_PATH: Path = Path('/kaggle/input/hull-tactical-market-prediction/')

# ============ RETURNS TO SIGNAL CONFIGS ============
MIN_SIGNAL: float = 0.0                                            # Minimum value for the daily signal 
MAX_SIGNAL: float = 2.0                                            # Maximum value for the daily signal 
SIGNAL_MULTIPLIER: float = 400.0                                   # Multiplier of the OLS market forward excess returns predictions to signal 

# ============ MODEL CONFIGS ============
CV = TimeSeriesSplit(n_splits = 5)                                 # Number of cross validation folds in the model fitting
L1_RATIO: np.ndarray = np.array([0.1, 0.5, 0.7, 0.9, 0.95])        # ElasticNet mixing parameter
ALPHAS: np.ndarray = np.logspace(-4, 2, 100)                       # Constant that multiplies the penalty terms
MAX_ITER: int = 1000000                                            # The maximum number of iterations
RANDOM_STATE: int = 17                                             # Random state

## Dataclasses Helpers

In [5]:
@dataclass
class DatasetOutput:
    X_train : pl.DataFrame 
    X_test: pl.DataFrame
    y_train: pl.Series
    y_test: pl.Series
    scaler: StandardScaler

@dataclass 
class ElasticNetParameters:
    l1_ratio : np.ndarray 
    cv: TimeSeriesSplit
    alphas: np.ndarray 
    max_iter: int
    random_state: int
    
    def __post_init__(self): 
        if self.l1_ratio.any() < 0 or self.l1_ratio.any() > 1: 
            raise ValueError("Wrong initializing value for ElasticNet l1_ratio")
        
@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier: float 
    min_signal : float = MIN_SIGNAL
    max_signal : float = MAX_SIGNAL

## Set the Parameters

In [6]:
ret_signal_params = RetToSignalParameters(
    signal_multiplier= SIGNAL_MULTIPLIER
)

enet_params = ElasticNetParameters(
    l1_ratio = L1_RATIO, 
    cv = CV, 
    alphas = ALPHAS, 
    max_iter = MAX_ITER,
    random_state = RANDOM_STATE
)

## Dataset Loading/Creating Helper Functions

In [7]:
def load_trainset() -> pl.DataFrame:
    """
    Loads and preprocesses the training dataset.

    Returns:
        pl.DataFrame: The preprocessed training DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "train.csv")
        .rename({'market_forward_excess_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
        .head(-10)
    )

def load_testset() -> pl.DataFrame:
    """
    Loads and preprocesses the testing dataset.

    Returns:
        pl.DataFrame: The preprocessed testing DataFrame.
    """
    return (
        pl.read_csv(DATA_PATH / "test.csv")
        .rename({'lagged_forward_returns':'target'})
        .with_columns(
            pl.exclude('date_id').cast(pl.Float64, strict=False)
        )
    )

def create_example_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """
    Creates new features and cleans a DataFrame.

    Args:
        df (pl.DataFrame): The input Polars DataFrame.

    Returns:
        pl.DataFrame: The DataFrame with new features, selected columns, and no null values.
    """

    vars_to_keep: List[str] = [
        "S2", "E19", "M17", "E11", "E18", "E12", "M18", "I2", 
        "E6", "M4", "I9", "E9", "E16", "I5", "M10", "P10",
        "E17", "M8", "E15", "M5", "M12", "P8", "S5", "U1", "U2"
    ]

    all_vars: List[str] = [
        'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9',
        'E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17',
        'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8',
        'E9', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M1',
        'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18',
        'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11',
        'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'S1',
        'S10', 'S11', 'S12', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8',
        'S9', 'V1', 'V10', 'V11', 'V12', 'V13', 'V2', 'V3', 'V4', 'V5',
        'V6', 'V7', 'V8', 'V9', 'U1', 'U2'
    ]

    
    return (
        df.with_columns(
            (pl.col("I2") - pl.col("I1")).alias("U1"),
            (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
        )
        .select(["date_id", "target"] + all_vars)
        .with_columns([
            pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
            for col in all_vars
        ])
        .drop_nulls()
    )
 
def join_train_test_dataframes(train: pl.DataFrame, test: pl.DataFrame) -> pl.DataFrame:
    """
    Joins two dataframes by common columns and concatenates them vertically.

    Args:
        train (pl.DataFrame): The training DataFrame.
        test (pl.DataFrame): The testing DataFrame.

    Returns:
        pl.DataFrame: A single DataFrame with vertically stacked data from common columns.
    """
    common_columns: list[str] = [col for col in train.columns if col in test.columns]
    
    return pl.concat([train.select(common_columns), test.select(common_columns)], how="vertical")

def split_dataset(train: pl.DataFrame, test: pl.DataFrame, features: list[str]) -> DatasetOutput: 
    """
    Splits the data into features (X) and target (y), and scales the features.

    Args:
        train (pl.DataFrame): The processed training DataFrame.
        test (pl.DataFrame): The processed testing DataFrame.
        features (list[str]): List of features to used in model. 

    Returns:
        DatasetOutput: A dataclass containing the scaled feature sets, target series, and the fitted scaler.
    """
    X_train = train.drop(['date_id','target']) 
    y_train = train.get_column('target')
    X_test = test.drop(['date_id','target']) 
    y_test = test.get_column('target')
    
    scaler = StandardScaler() 
    
    X_train_scaled_np = scaler.fit_transform(X_train)
    X_train = pl.from_numpy(X_train_scaled_np, schema=features)
    
    X_test_scaled_np = scaler.transform(X_test)
    X_test = pl.from_numpy(X_test_scaled_np, schema=features)
    
    
    return DatasetOutput(
        X_train = X_train,
        y_train = y_train, 
        X_test = X_test, 
        y_test = y_test,
        scaler = scaler
    )

## Converting Return Prediction to Signal

Here is an example of a potential function used to convert a prediction based on the market forward excess return to a daily signal position. 

In [8]:
def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params: RetToSignalParameters
) -> np.ndarray:
    """
    Converts raw model predictions (expected returns) into a trading signal.

    Args:
        ret_arr (np.ndarray): The array of predicted returns.
        params (RetToSignalParameters): Parameters for scaling and clipping the signal.

    Returns:
        np.ndarray: The resulting trading signal, clipped between min and max values.
    """
    return np.clip(
        ret_arr * params.signal_multiplier + 1, params.min_signal, params.max_signal
    )

## Looking at the Data

In [9]:
train: pl.DataFrame = load_trainset()
test: pl.DataFrame = load_testset() 
print(train.tail(3)) 
print(test.head(3))

shape: (3, 98)
┌─────────┬─────┬─────┬─────┬───┬───────────┬─────────────────┬────────────────┬──────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ V9        ┆ forward_returns ┆ risk_free_rate ┆ target   │
│ ---     ┆ --- ┆ --- ┆ --- ┆   ┆ ---       ┆ ---             ┆ ---            ┆ ---      │
│ i64     ┆ f64 ┆ f64 ┆ f64 ┆   ┆ f64       ┆ f64             ┆ f64            ┆ f64      │
╞═════════╪═════╪═════╪═════╪═══╪═══════════╪═════════════════╪════════════════╪══════════╡
│ 8977    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.708599 ┆ 0.004187        ┆ 0.000162       ┆ 0.003713 │
│ 8978    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.725858 ┆ 0.002279        ┆ 0.000162       ┆ 0.001805 │
│ 8979    ┆ 0.0 ┆ 0.0 ┆ 0.0 ┆ … ┆ -0.720092 ┆ 0.003541        ┆ 0.000161       ┆ 0.003068 │
└─────────┴─────┴─────┴─────┴───┴───────────┴─────────────────┴────────────────┴──────────┘
shape: (3, 99)
┌─────────┬─────┬─────┬─────┬───┬───────────┬───────────┬─────────────────────┬────────────────────┐
│ date_id ┆ D1  ┆ D2  ┆ D3  ┆ … ┆ is_scor

## Generating the Train and Test

In [9]:
df: pl.DataFrame = join_train_test_dataframes(train, test)
df = create_example_dataset(df=df) 
train: pl.DataFrame = df.filter(pl.col('date_id').is_in(train.get_column('date_id')))
test: pl.DataFrame = df.filter(pl.col('date_id').is_in(test.get_column('date_id')))

FEATURES: list[str] = [col for col in test.columns if col not in ['date_id', 'target']]

dataset: DatasetOutput = split_dataset(train=train, test=test, features=FEATURES) 

X_train: pl.DataFrame = dataset.X_train
X_test: pl.DataFrame = dataset.X_test
y_train: pl.DataFrame = dataset.y_train
y_test: pl.DataFrame = dataset.y_test
scaler: StandardScaler = dataset.scaler 

## Fitting the Model 

In [10]:
model_cv: ElasticNetCV = ElasticNetCV(
    **asdict(enet_params)
)
model_cv.fit(X_train, y_train) 
        
# Fit the final model using the best alpha found by cross-validation
model: ElasticNet = ElasticNet(
                               alpha=model_cv.alpha_, 
                               l1_ratio=model_cv.l1_ratio_, 
                               random_state=enet_params.random_state
                              ) 
model.fit(X_train, y_train)

ElasticNet(alpha=0.008697490026177835, l1_ratio=0.1, random_state=17)

In [11]:
X_train

D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,E19,E2,E20,E3,E4,E5,E6,E7,E8,E9,I1,I2,I3,I4,I5,I6,I7,I8,…,P12,P13,P2,P3,P4,P5,P6,P7,P8,P9,S1,S10,S11,S12,S2,S3,S4,S5,S6,S7,S8,S9,V1,V10,V11,V12,V13,V2,V3,V4,V5,V6,V7,V8,V9,U1,U2
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.179836,-0.179836,-0.223899,-1.166331,-0.485816,-1.7835,-0.2202,-0.409668,-0.40884,-1.54657,-1.418971,-0.417604,-0.150749,-0.023499,1.505327,-1.090225,1.082366,1.055233,0.676926,-0.890033,0.560007,0.079306,-0.317338,-0.530454,1.168061,-0.038188,0.555994,-0.60968,-1.165796,1.203027,-1.052132,0.265737,-1.256147,1.361055,1.052328,0.456792,-1.344915,…,-0.086405,-1.329807,0.423753,-0.971652,1.439308,-0.198494,-0.340755,-0.632392,0.419509,-0.876069,-0.050324,1.002948,-0.144149,-0.745131,-0.841899,-0.913667,-1.268816,-0.111638,-1.691948,-0.074318,0.196106,1.550459,1.729836,-0.930746,1.801591,2.035521,-0.564032,1.841738,1.72589,1.803183,0.082593,1.759323,-0.899521,1.637143,-0.854984,-1.14379,-0.407136
-0.179836,-0.179836,-0.223899,-1.166331,-0.485816,0.560695,-0.2202,2.440999,-0.40884,-1.548156,-1.420258,-0.435737,-0.175893,-0.063993,1.373486,-1.208496,1.081224,1.05518,0.677827,-0.814949,0.566041,0.072229,-0.310368,-0.53974,1.168961,-0.046835,0.555309,-0.609055,-1.166858,1.165341,-1.050689,0.093415,-1.258138,1.364832,1.106411,0.267757,-1.404589,…,2.57979,-1.574515,0.427939,0.569313,-0.307687,-0.381062,-0.025405,-0.616039,0.41875,-0.810625,-0.061337,0.225582,-0.121277,0.278901,-1.098594,-0.736139,-0.857048,-0.144468,-1.656277,-0.073141,0.193821,1.152405,1.77731,-0.913262,1.758353,1.818037,-0.519361,1.841738,1.759924,1.805467,-0.121512,1.311241,-0.846415,1.608607,-0.835622,-1.137037,-0.430667
-0.179836,-0.179836,-0.223899,-1.166331,2.058392,0.560695,-0.2202,2.440999,-0.40884,-1.549741,-1.421546,-0.45387,-0.201037,-0.104487,1.241644,-1.209474,1.080084,1.055127,0.67873,-0.877987,0.612661,0.065174,-0.258859,-0.549026,1.169861,-0.055481,0.554625,-0.608434,-1.167919,1.091317,-1.066903,0.173539,-0.541419,1.325548,1.075989,0.598568,-1.357573,…,3.434513,-1.575637,0.45819,1.668375,-1.322587,-0.503006,0.048141,-0.007051,0.427107,-0.586246,-0.103838,0.200904,-0.30217,0.404231,-0.908827,-0.529964,-1.067134,-0.082515,-1.657466,0.529684,0.202923,1.431476,1.750182,-0.900258,1.72877,1.320423,-0.564493,1.841738,1.678243,1.812321,0.227638,1.485353,-0.916216,1.549422,-0.832619,-1.141347,-0.424061
-0.179836,-0.179836,-0.223899,-1.166331,2.058392,0.560695,4.541328,2.440999,-0.40884,-1.551325,-1.422833,-0.472003,-0.22618,-0.144981,1.109803,-1.210451,1.060881,1.806476,0.679634,-0.860787,0.671067,0.068907,-0.1939,-0.558311,1.17076,-0.064128,0.555818,-0.587945,-1.168981,1.223215,-1.047611,0.107684,-0.543409,1.266616,0.727834,0.182691,-1.393739,…,2.10664,-1.007646,0.209985,1.207111,-0.331634,-0.4551,-0.151239,0.568774,0.434413,-0.576897,-0.097273,-1.154346,-1.273166,2.279591,-1.038394,-0.333293,-1.359153,-0.244666,-1.571856,0.530861,0.154145,1.256245,1.695926,-0.92251,1.751526,1.296783,-0.62322,1.841738,1.621521,1.814605,0.121862,1.46743,-1.007486,1.553649,-0.8847,-1.142523,-0.445823
-0.179836,-0.179836,-0.223899,-1.166331,2.058392,0.560695,-0.2202,-0.409668,2.445943,-1.552909,-1.42412,-0.490136,-0.251324,-0.185475,0.977961,-1.211428,1.059779,1.803822,0.68054,-0.989555,0.688803,0.114145,-0.173236,-0.567597,1.17166,-0.072774,0.478639,-0.587358,-1.170043,1.032097,-1.099321,0.137319,-1.260129,1.276297,0.717694,0.442614,-1.379273,…,1.670773,-0.93805,0.227998,-0.379411,1.263696,0.090608,-0.34066,0.387561,0.434663,-0.5395,-0.098571,1.231222,0.710411,-0.481942,-0.65622,-0.233461,-0.186873,-0.448836,-1.460087,0.532039,0.226586,0.28166,1.702708,-0.94328,1.715116,1.143125,-0.660298,1.834539,1.671437,1.812321,0.505034,0.530298,-1.074573,1.

In [12]:
X_test

D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,E19,E2,E20,E3,E4,E5,E6,E7,E8,E9,I1,I2,I3,I4,I5,I6,I7,I8,…,P12,P13,P2,P3,P4,P5,P6,P7,P8,P9,S1,S10,S11,S12,S2,S3,S4,S5,S6,S7,S8,S9,V1,V10,V11,V12,V13,V2,V3,V4,V5,V6,V7,V8,V9,U1,U2
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
-0.179836,-0.179836,-0.223899,-1.166331,2.058392,0.560695,-0.2202,2.440999,-0.40884,-0.172844,-0.761104,-0.780267,-0.653622,-0.83338,-1.131503,1.58994,-0.266114,-0.440484,0.042682,-0.697011,0.198406,-0.054193,0.686671,0.258818,-0.590988,-0.012249,-0.004975,-0.013902,1.702587,-1.299003,0.678928,-1.469559,0.945775,-0.336177,0.190391,-1.126377,0.541127,…,0.881502,0.938793,-0.254867,-0.43408,1.525974,0.622295,-0.499255,0.149327,0.172981,-0.586246,-0.249345,-0.000636,-0.148307,-0.120247,0.038841,-0.293118,-0.852846,-0.102003,-1.065329,-0.106108,0.497543,0.795454,-0.117112,-0.713407,0.775269,1.138397,-0.470503,1.416988,1.435471,1.220644,0.459732,1.275395,-0.816238,-0.945858,-0.783167,0.81327,-0.147618
-0.179836,-0.179836,-0.223899,-1.166331,2.058392,0.560695,-0.2202,2.440999,-0.40884,-0.176632,-0.762391,-0.7984,-0.678766,-0.873874,-1.263345,1.590917,-0.265845,-0.440108,0.042041,-0.548827,0.140983,-0.056552,0.628192,0.249533,-0.591888,-0.020896,0.057241,-0.013911,1.703649,-1.300349,0.655965,-1.465169,0.943784,-0.345421,0.291795,-1.127953,0.544744,…,-0.418471,0.65592,-0.203304,1.634207,-0.333915,0.718753,-0.441046,-0.499843,0.152479,-0.530151,-0.241621,0.383934,0.450509,1.601884,0.229993,-0.10566,-0.294017,0.076024,-1.008255,-0.106108,0.552467,1.215142,0.739673,-0.660932,0.535187,0.779075,-0.392812,1.347397,0.772952,1.232066,0.558103,0.48677,-0.726282,-1.038863,-0.714543,0.792277,-0.144896
-0.179836,-0.179836,-0.223899,-1.166331,2.058392,0.560695,-0.2202,-0.409668,2.445943,-0.180409,-0.763678,-0.816533,-0.703909,-0.914368,-1.395186,1.591895,0.123338,-0.343105,-0.018797,-0.461122,0.079613,-0.059042,0.565838,0.240247,-0.592788,-0.029542,0.036513,-0.013921,1.70471,-1.356877,0.688294,-1.469559,0.941793,-0.349215,0.315456,-1.269729,0.515811,…,0.371737,-1.036831,-0.174264,-0.92154,0.522478,-1.020383,-0.517334,-1.213317,0.12904,-0.548849,-0.278661,-1.345602,-1.073561,-0.952523,0.293084,-0.009978,-1.281422,0.252962,-1.084353,-0.106108,0.551341,0.907947,0.547518,-0.587281,0.672864,0.797987,-0.231982,1.292203,0.595977,1.236635,0.532655,0.909247,-0.551788,-0.854967,-0.602636,0.830237,-0.150569
-0.179836,-0.179836,-0.223899,-1.166331,2.058392,0.560695,-0.2202,-0.409668,2.445943,-0.184174,-0.764966,0.23519,0.754422,-0.185475,0.977961,1.592872,0.123302,-0.342832,-0.019414,-0.578285,0.118672,-0.061521,0.604938,0.230962,-0.593688,-0.038188,0.036729,-0.013931,1.705772,-1.45782,0.720571,-1.454193,0.939802,-0.353909,0.335737,-1.250825,0.494112,…,0.37199,0.484175,-0.226306,0.848349,-0.922329,-0.070472,-0.333216,0.058074,0.151752,-0.530151,-0.322315,-0.592915,-0.882272,-0.635526,-0.092584,0.056576,-1.020915,0.114618,-1.181854,-0.106108,0.537175,-0.167233,0.651508,-0.633631,0.399785,0.637237,-0.307192,1.210613,-0.096037,1.248057,0.493682,0.635277,-0.630468,-0.975451,-0.651354,0.874539,-0.120792
-0.179836,-0.179836,-0.223899,-1.166331,-0.485816,0.560695,4.541328,-0.409668,2.445943,-0.187928,-0.766253,0.217057,0.729278,-0.225969,0.84612,1.59385,0.123265,-0.342558,-0.020031,-0.7154,0.184967,-0.068327,0.67137,0.221676,-0.594588,-0.046835,0.034443,-0.008691,1.706834,-1.483392,0.669926,-1.427851,0.937811,-0.36041,0.342497,-1.209868,0.504961,…,0.647335,-0.649564,-0.243259,0.457698,-0.198215,0.219458,-0.274326,0.859276,0.174948,-0.446009,-0.300184,0.215299,-0.152466,1.016226,-0.222314,-0.051251,-0.54612,0.002323,-1.315025,-0.106108,0.531214,-0.037432,0.429965,-0.663736,-0.045106,0.381929,-0.393683,1.143421,0.802447,1.261764,0.415419

In [13]:
preds = model.predict(X_train)
r2 = r2_score(y_train, preds)
rmse = np.sqrt(mean_squared_error(y_train, preds))
print(r2)
print(rmse)
print(model.coef_)
print(FEATURES)

6.857585168884572e-05
0.010836267785343563
[ 0.00000000e+00  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00 -0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00 -0.00000000e+00
 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00  0.00000000e+00
 -0.00000000e+00 -0.00000000e+00 -0.00000000e+00  0.00000000e+00
  0.00000000e+00 -0.00000000e+00 -0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00 -0.00000000e+00  0.00000000e+00
 -0.00000000e+00 -0.00000000e+00  0.00000000e+00  0.00000000e+00
 -0.00000000e+00 -0.00000000e+00  0.00000000e+00 -0.00000000e+00
  0.00000000e+00 -0.00000000e+00  0.00000000e+00  0.00000000e+00
 -0.00000000e+00 -0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00 -0.00000000e+00 -0.00000000e+00
  0.00000000e+00  0.00000000e+00 -0.00000000e+00  0.00000000e+00
  0.00000000e+00 -4.61707909e-06 -0.00000000e+0

## Prediction Function via Kaggle Server

In [14]:
def predict(test: pl.DataFrame) -> float:
    test = test.rename({'lagged_forward_returns':'target'})
    df: pl.DataFrame = create_example_dataset(test)
    X_test: pl.DataFrame = df.select(FEATURES)
    X_test_scaled_np: np.ndarray = scaler.transform(X_test)
    X_test: pl.DataFrame = pl.from_numpy(X_test_scaled_np, schema=FEATURES)
    raw_pred: float = model.predict(X_test)[0]
    return convert_ret_to_signal(raw_pred, ret_signal_params)

## Launch Server

In [15]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))